## Exercise 4 - Duration, convexity, and a DV01 hedge.

Core part: choose a coupon bond, or define one (for example 10 years, 4% annual coupon). Build its cash-flow table as in Section 0. Compute its price, Macaulay duration, modified duration and DV01. Then change the yield by ±25, ±100 and ±300 bp, reprice the bond exactly each time, and compare the true price change with the estimate from duration alone (`−ModDur × Δy`). Show at what size of yield change the duration estimate becomes inaccurate.


   Extension: add the convexity term and check that `ΔP/P ≈ −ModDur × Δy + ½ × Convexity × (Δy)²` removes most of the error at ±300 bp. Then hedge the bond with a Treasury futures contract (or a proxy), sized so that the bond and the hedge have the same DV01. Report the remaining profit or loss of the hedged position under the same yield changes.




#### (1) Report

This analysis aims to check how changes in bond's yield affect its price. Exact price changes are compared with the ones estimated using duration alone and duration plus convexity. The chosen obligation has a par value of $100, a 5% annual coupon and 10 years to maturity. The assumed annual yield is 6%. The proxy bond has a face value of $100, coupon of 4%, and its maturity is 5 years. 

The original bond pays 5 at the end of each year and repays its par value at maturity, making the final payment 105. Valuation takes place on a coupon date, so there is no accrued interest that would have to be accounted in. The bond's price is $92.64, and is the sum of all present values of inflows at time 0. 

| Measure           | Value        |
| ----------------- | -----------: |
| Price             | 92.6399      |
| Macaulay duration | 8.0225 years |
| Modified duration | 7.5684       |
| DV01              | 0.070114     |
| Convexity         | 72.5693      |

The bond’s price of $92.64 is below its face value of 100 because its 5% coupon is lower than the market yield of 6%. Its Macaulay duration of 8.02 years represents the average timing of its payments, weighted by their present values. The modified duration of 7.57 means that a 1 percentage-point increase in yield would approximately reduce its price by 7.57%, while its DV01 of 0.07 means that a 1 bps increase would approximately reduce its value by 0.07 monetary units. The convexity term of 72.57 captures the curvature of the price yield relationship and improves the duration estimate, as this realtion is not linear. 

| Measure                           | Value      |
| :-------------------------------: | :--------: |
| Proxy bond DV01                    | 0.039832   |
| Proxy bonds to short per bond held | 1.7602     |
| Net initial DV01                  | 0.00000000 |

From the above table it can be seen that shorting 1.7602 proxy bonds offsets the initial interest rate sensitivity of one original bond. The zero net initial DV01 means that for a small, equal increase in both yields, the bond’s loss should be approximately offset by the hedge’s gain and vice versa.

The bond was tested for yield changes of +/-25, +/-100 and +/-300 bps from the original yield of 6%. Its price increased when yields fell and decreased when yields rose. For example, a 300 bp decrease raised the price to $117.06, while a 300 bp increase lowered it to $74.33. Duration gave close estimates for small yield changes, but became less accurate for larger changes. At +/-300 bp, its errors were around 3 percentage points. Adding convexity reduced both errors to less than 0.4 percentage points, making the estimates much closer to the actual changes.

The introduced proxy bond hedge reduced the effect of yield changes on the position. For example, when yields rose by 300 bp, the original bond lost $18.31, but the combined bond and hedge made a small profit of $1.09. However, the hedge was not perfect because the two bonds respond differently to larger yield changes. These results assume that both yields move by the same amount and exclude borrowing costs and trading fees. fees. 


#### (2) Code

In [12]:
import pandas as pd
import numpy as np

face_value = 100
coupon = 0.05
base_yield = 0.06
n = 10

# Annual coupons
def bond_table(face_value, coupon, n, bond_yield):
    table = pd.DataFrame({
        "years": np.arange(1, n + 1),
        "cash flow": coupon * face_value,
    })
    table.loc[table.index[-1], "cash flow"] += face_value
    table["discount factor"] = (1 + bond_yield) ** (-table["years"])
    table["present value"] = table["cash flow"] * table["discount factor"]
    return table

cash_flows = bond_table(face_value, coupon, n, base_yield)
bond_price = cash_flows["present value"].sum()
macaulay_d = (cash_flows["years"] * cash_flows["present value"]).sum() / bond_price
modified_d = macaulay_d / (1 + base_yield)
dv01 = bond_price * modified_d * 0.0001

convexity = (
    cash_flows["years"] * (cash_flows["years"] + 1)
    * cash_flows["present value"]
).sum() / (bond_price * (1 + base_yield) ** 2)

print("CASH FLOW TABLE: ")
print(cash_flows.round(4).to_string(index=False))
print(f"\nPrice: {bond_price:.4f}")
print(f"Macaulay duration: {macaulay_d:.4f}")
print(f"Modified duration: {modified_d:.4f}")
print(f"DV01: {dv01:.6f}")
print(f"Convexity: {convexity:.4f}")

# Proxy hedge: short a 5-year, 4% annual coupon bond, face value 100.
# Both yields start at 6% and move by the same amount in every scenario.
hedge_table = bond_table(100, 0.04, 5, base_yield)
hedge_price = hedge_table["present value"].sum()
hedge_duration = (
    hedge_table["years"] * hedge_table["present value"]
).sum() / (hedge_price * (1 + base_yield))
hedge_dv01 = hedge_price * hedge_duration * 0.0001
hedge_units = dv01 / hedge_dv01
print(f"\nProxy bond DV01: {hedge_dv01:.6f}")
print(f"Short {hedge_units:.4f} proxy bonds per one bond held")
print(f"Net initial DV01: {dv01 - hedge_units * hedge_dv01:.8f}")

results = []
for bp in [-300, -100, -25, 25, 100, 300]:
    dy = bp / 10000
    new_yield = base_yield + dy
    new_price = bond_table(face_value, coupon, n, new_yield)["present value"].sum()
    true_change = (new_price - bond_price) / bond_price
    duration_change = -modified_d * dy
    convexity_change = duration_change + 0.5 * convexity * dy ** 2

    new_hedge_price = bond_table(100, 0.04, 5, new_yield)["present value"].sum()
    bond_pnl = new_price - bond_price
    hedge_pnl = -hedge_units * (new_hedge_price - hedge_price)
    results.append({
        "shock bp": bp,
        "price": new_price,
        "true %": 100 * true_change,
        "duration %": 100 * duration_change,
        "dur+conv %": 100 * convexity_change,
        "dur error pp": 100 * (duration_change - true_change),
        "dur+conv error pp": 100 * (convexity_change - true_change),
        "bond P&L": bond_pnl,
        "hedged P&L": bond_pnl + hedge_pnl,
    })

results = pd.DataFrame(results)
print("\nSCENARIOS FOR DIFFERENT BPS CHANGE")
display(round(results,4))


CASH FLOW TABLE: 
 years  cash flow  discount factor  present value
     1        5.0           0.9434         4.7170
     2        5.0           0.8900         4.4500
     3        5.0           0.8396         4.1981
     4        5.0           0.7921         3.9605
     5        5.0           0.7473         3.7363
     6        5.0           0.7050         3.5248
     7        5.0           0.6651         3.3253
     8        5.0           0.6274         3.1371
     9        5.0           0.5919         2.9595
    10      105.0           0.5584        58.6315

Price: 92.6399
Macaulay duration: 8.0225
Modified duration: 7.5684
DV01: 0.070114
Convexity: 72.5693

Proxy bond DV01: 0.039832
Short 1.7602 proxy bonds per one bond held
Net initial DV01: 0.00000000

SCENARIOS FOR DIFFERENT BPS CHANGE


,shock bp,price,true %,duration %,dur+conv %,dur error pp,dur+conv error pp,bond P&L,hedged P&L
0,-300,117.0604,26.3607,22.7053,25.9709,-3.6554,-0.3898,24.4205,1.5294
1,-100,100.0000,7.9448,7.5684,7.9313,-0.3764,-0.0136,7.3601,0.1514
2,-25,94.4140,1.9150,1.8921,1.9148,-0.0229,-0.0002,1.7740,0.0091
3,25,90.9079,-1.8696,-1.8921,-1.8694,-0.0225,0.0002,-1.7320,0.0088
4,100,85.9528,-7.2184,-7.5684,-7.2056,-0.3501,0.0128,-6.6871,0.1354
5,300,74.3294,-19.7653,-22.7053,-19.4397,-2.9400,0.3256,-18.3105,1.0936
